# Workflow for downloading stallar streams data from the Ibata ert al. 2024 catalog available on VizieR
Quickly looking at the data and keeping only the stars that have a complete &D phase space 

In [ ]:
import pandas as pd
import numpy as np
from astropy.io import ascii

In [11]:
data_table = ascii.read('Ibata_catalog.txt', format='mrt')
df = data_table.to_pandas()
print(f"Data Loaded : {len(df)} stars.")
print(df.columns)

Data Loaded : 24540 stars.
Index(['Gaia', 'RAdeg', 'DEdeg', 'plx', 'pmRA', 'pmDE', 'Gmag', '(B-R)',
       'VHel', 'e_VHel', 'r_VHel', 'Stream'],
      dtype='object')


In [16]:
# Filtering on the 6D space (RA, Dec, pmRA, pmDec, Vh, plx)
df_6d = df[df['VHel'] != 0.0].copy()
df_6d = df_6d[df_6d['plx'] > 0].copy()
print(f"Stars with complete 6D phase space : {len(df_6d)}")

Stars with complete 6D phase space : 1703


In [17]:
# making the variables ready for galpy
# RA, Dec (degrees)
ra = df_6d['RAdeg'].values
dec = df_6d['DEdeg'].values

# Proper motions (in mas/yr)
pmra = df_6d['pmRA'].values
pmdec = df_6d['pmDE'].values

# Radial velocity (in km/s)
vlos = df_6d['VHel'].values

# Conversion of the parallax (mas) to distance (kpc)
# Formula : d(kpc) = 1 / parallax(mas)
distance_kpc = 1.0 / df_6d['plx'].values

# Creation of the final table
df_final = pd.DataFrame({
    'RA': ra,
    'Dec': dec,
    'Distance_kpc': distance_kpc,
    'pmRA': pmra,
    'pmDec': pmdec,
    'Vlos': vlos,
    'Stream_ID': df_6d['Stream'].values # To keep track of the original cluster
})

print(f"Stars ready for 6D integration : {len(df_final)} stars.")
print(df_final.head())

Stars ready for 6D integration : 1703 stars.
           RA        Dec  Distance_kpc   pmRA  pmDec    Vlos  Stream_ID
0    0.714213   7.207192     11.111111 -0.553  1.405 -115.03          1
1  355.371856  15.482531     16.666667 -0.844  1.699  -90.95          1
2  357.217906  12.842164    125.000000 -0.744  1.570 -116.37          1
3  359.805780   8.925398    250.000000 -0.555  1.446 -115.52          1
4  359.854237   8.529315     10.526316 -0.607  1.428 -115.22          1


In [18]:
# Saving for the next step with galpy
df_final.to_csv("stars_6D_ibata2024.csv", index=False)
print("File 'stars_6D_ibata2024.csv' saved successfully !")

File 'stars_6D_ibata2024.csv' saved successfully !
